# 01 — Market, VRP, OU z, Markov regimes

Read-only notebook. Run `python scripts/run_daily_market_update.py` first so DuckDB (or `data/processed/*.parquet`) exists.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
from config import DUCKDB_PATH, PROCESSED_DATA_PATH
from data_pipeline.db_utils import DatabaseManager

def load(name):
    pq = PROCESSED_DATA_PATH / f"{name}.parquet"
    csv = PROCESSED_DATA_PATH / f"{name}.csv"
    if pq.exists():
        return pd.read_parquet(pq)
    if csv.exists():
        return pd.read_csv(csv)
    db = DatabaseManager(DUCKDB_PATH)
    return db.load_table(name).to_pandas()

reg = load("regime_predictions")
reg["date"] = pd.to_datetime(reg["date"])
reg = reg.sort_values("date")
print(reg.tail())
print("rows", len(reg), "cols", list(reg.columns))


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(reg["date"], reg["indiavix"], color="#2b6cb0", lw=1.2, label="INDIAVIX")
if "prob_regime_high_vol" in reg.columns:
    storm = reg["indiavix"].where(reg["prob_regime_high_vol"] >= 0.5)
    ax[0].plot(reg["date"], storm, color="#c53030", lw=1.6, label="high-vol window")
ax[0].set_ylabel("India VIX")
ax[0].legend()
ax[0].set_title("VIX and high-vol windows")
if "prob_regime_high_vol" in reg.columns:
    ax[1].plot(reg["date"], reg["prob_regime_high_vol"], color="#c53030", lw=1.0)
    ax[1].axhline(0.5, color="grey", ls="--", lw=0.8)
    ax[1].set_ylabel("P(high vol)")
ax[1].set_xlabel("Date")
plt.tight_layout()
plt.show()


In [ ]:
feat = None
try:
    feat = load("market_features")
    feat["date"] = pd.to_datetime(feat["date"])
    print(feat[["date", "indiavix", "realized_vol_21d", "vrp", "ou_z_score"]].tail())
except Exception as exc:
    print("market_features not available:", exc)
